# 1. Import Libraries

In [28]:
import pandas as pd
import numpy as np

# 2. Load Dataset

In [29]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df = pd.read_csv('dataset.csv')

/kaggle/input/datasets/mayankbulchandani/loan-approval-dataset/dataset.csv


# 3. Exploratory Data Analysis

In [31]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,lp001002,male,no,0,graduate,no,5849,0.0,NaN,360.0,1.0,urban,y
1,lp001003,male,yes,1,graduate,no,4583,1508.0,128.0,360.0,1.0,rural,n
2,lp001005,male,yes,0,graduate,yes,3000,0.0,66.0,360.0,1.0,urban,y
3,lp001006,male,yes,0,not graduate,no,2583,2358.0,120.0,360.0,1.0,urban,y
4,lp001008,male,no,0,graduate,no,6000,0.0,141.0,360.0,1.0,urban,y


In [32]:
df.corr(numeric_only=True)

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
ApplicantIncome,1.000000,-0.116605,0.570909,-0.045306,-0.014715
CoapplicantIncome,-0.116605,1.000000,0.188619,-0.059878,-0.002056
LoanAmount,0.570909,0.188619,1.000000,0.039447,-0.008433
Loan_Amount_Term,-0.045306,-0.059878,0.039447,1.000000,0.001470
Credit_History,-0.014715,-0.002056,-0.008433,0.001470,1.000000


In [33]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [34]:
df.isna().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [35]:
df['Loan_Amount_Term'].value_counts()

Loan_Amount_Term
360.0    512
180.0     44
480.0     15
300.0     13
84.0       4
240.0      4
120.0      3
60.0       2
36.0       2
12.0       1
Name: count, dtype: int64

In [36]:
df = df.drop(columns = "Loan_ID")

In [37]:
df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,male,no,0,graduate,no,5849,0.0,NaN,360.0,1.0,urban,y
1,male,yes,1,graduate,no,4583,1508.0,128.0,360.0,1.0,rural,n
2,male,yes,0,graduate,yes,3000,0.0,66.0,360.0,1.0,urban,y
3,male,yes,0,not graduate,no,2583,2358.0,120.0,360.0,1.0,urban,y
4,male,no,0,graduate,no,6000,0.0,141.0,360.0,1.0,urban,y


In [38]:
df['Married'].mode()

0    yes
Name: Married, dtype: object

# 4. Train-Test Split

In [43]:
from sklearn.model_selection import train_test_split

# Note: X and y are defined after preprocessing below, but we do the split later.
# To keep the original flow, we keep the split here but it will be overridden.
# Actually we'll move split to after preprocessing in the next section.

# 5. Data Preprocessing

In [39]:
# Fill missing values
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

# Feature engineering
df['TotalIncome'] = np.log(df['ApplicantIncome']+df['CoapplicantIncome'])
df['EMI'] = df['LoanAmount']/df['Loan_Amount_Term']
df['Income_to_Loan'] = df['TotalIncome'] / df['LoanAmount']

# Encode target and clean Dependents
df['Loan_Status'] = df['Loan_Status'].map({'n': 0, 'y': 1})
df["Dependents"] = df['Dependents'].replace("3+","3")

# Drop original income columns
df = df.drop(columns = ['ApplicantIncome','CoapplicantIncome','TotalIncome'])

# Verify no missing values
print(df.isna().sum())

,Gender,Married,Dependents,Education,Self_Employed,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,EMI,Income_to_Loan
0,male,no,0,graduate,no,128.0,360.0,1.0,urban,y,0.355556,0.067766
1,male,yes,1,graduate,no,128.0,360.0,1.0,rural,n,0.355556,0.068083
2,male,yes,0,graduate,yes,66.0,360.0,1.0,urban,y,0.183333,0.121309
3,male,yes,0,not graduate,no,120.0,360.0,1.0,urban,y,0.333333,0.070878
4,male,no,0,graduate,no,141.0,360.0,1.0,urban,y,0.391667,0.061699
...,...,...,...,...,...,...,...,...,...,...,...,...
609,female,no,0,graduate,no,71.0,360.0,1.0,rural,y,0.197222,0.112288
610,male,yes,3+,graduate,no,40.0,180.0,1.0,rural,y,0.222222,0.208005
611,male,yes,1,graduate,no,253.0,360.0,1.0,urban,y,0.702778,0.035674
612,male,yes,2,graduate,no,187.0,360.0,1.0,urban,y,0.519444,0.047774


In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Split again after preprocessing
X = df.drop(columns = 'Loan_Status')
y = df['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Column transformer
transformer = ColumnTransformer(transformers=[
    ('tnf1', OneHotEncoder(sparse_output=False, drop='first'), ['Property_Area','Gender','Married','Education','Self_Employed','Dependents']),
    ('tnf3', StandardScaler(), ['LoanAmount', 'Loan_Amount_Term','EMI','Income_to_Loan'])
], remainder='passthrough')

X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

X_train_transformed_df = pd.DataFrame(X_train_transformed)
X_test_transformed_df = pd.DataFrame(X_test_transformed)

In [49]:
X_train_transformed_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.131588,0.280691,-0.620918,1.594205,3254.0,0.0,1.0,8.087640
1,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,-0.592309,0.280691,-0.391569,0.162054,3315.0,0.0,1.0,8.106213
2,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.040758,0.280691,-0.122334,-0.349064,3340.0,1710.0,0.0,8.527144
3,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,-0.393010,-2.486768,0.256589,-0.036318,2653.0,1500.0,0.0,8.331586
4,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.040758,0.280691,-0.122334,-0.354232,2620.0,2223.0,1.0,8.485290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.029583,0.280691,-0.152249,-0.288220,2971.0,2791.0,1.0,8.659040
487,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.474526,0.280691,0.062141,-0.501566,2625.0,6250.0,1.0,9.090994
488,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.287499,0.280691,-0.261938,-0.107320,2799.0,2253.0,1.0,8.527539
489,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.111647,0.280691,-0.187150,-0.256399,2484.0,2302.0,1.0,8.473450


# 6. Logistic Regression

In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score

In [51]:
lor = LogisticRegression(random_state = 42 , max_iter = 10000)
lor.fit(X_train_transformed_df , y_train)

y_pred = lor.predict(X_test_transformed_df)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}\n")

# 7. Random Forest

In [52]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train_transformed_df, y_train)

y_pred = rf.predict(X_test_transformed_df)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}\n")

# 8. XGBoost

In [53]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train_transformed, y_train)

y_pred = xgb.predict(X_test_transformed)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}\n")

# 9. Stacking Classifier

In [54]:
from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[
        ('lor', lor),
        ('rf', rf),
        ('xgb', xgb)
    ],
    final_estimator=LogisticRegression()
)

stack.fit(X_train_transformed, y_train)

# Note: y_pred from previous cell is reused; we should predict with stack
y_pred_stack = stack.predict(X_test_transformed)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_stack):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_stack):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_stack):.4f}\n")

# 10. Model Comparison & Conclusion

| Model               | Accuracy | Recall   | F1 Score |
|---------------------|----------|----------|----------|
| Logistic Regression | 0.8618   | 0.9882   | 0.9081   |
| Random Forest       | 0.8374   | 0.9176   | 0.8864   |
| XGBoost             | 0.8130   | 0.9294   | 0.8729   |
| Stacking Classifier | 0.8130   | 0.9294   | 0.8729   |

**Conclusion**:  
The Logistic Regression model achieves the highest accuracy (86.18%) and F1 score (0.908), making it the most suitable model for this dataset. Its high recall (0.988) indicates that it correctly identifies almost all loan approvals, which is crucial for minimising false negatives (rejecting loans that should be approved). The tree‑based ensemble methods (Random Forest and XGBoost) perform reasonably well but lag behind logistic regression, possibly due to the relatively small dataset and the limited number of features. The stacking classifier did not improve over XGBoost, suggesting that the base models are already well‑tuned or that the dataset does not benefit from stacking. Future improvements could involve hyperparameter tuning, feature engineering (e.g., debt‑to‑income ratio), and exploring other algorithms like SVM or neural networks.